# Happy — обучение с нуля на Kaggle

Этот ноутбук **не содержит логики обучения**: он клонирует репозиторий и запускает
`scripts/train.py`. Вся логика — в репозитории, чтобы тот же код можно было проверить.

**Ничего не потеряется, даже если сессия умрёт:**
1. чекпоинт после каждой эпохи лежит в `/kaggle/working/artifacts/checkpoints/`;
2. он же сразу уходит в Hugging Face (`{run}/checkpoints/...`);
3. в конце — `manifest.json` с sha256 каждого файла и `latest.json`.

Перед запуском добавь секреты (Add-ons → Secrets):
- `HF_TOKEN` — токен Hugging Face со scope **write**;
- `HF_REPO` — например `vaprooll/Happy`;
- `GH_REPO` — например `qtttyr/Happy` (публичный репозиторий);
- `GH_TOKEN` — только если репозиторий приватный.


In [ ]:
!pip install -q pymorphy3 huggingface_hub tqdm


In [ ]:
import os, subprocess, sys
from kaggle_secrets import UserSecretsClient

SMOKE = True   # первый прогон: True (проверка HF-цепочки). Полный: False

s = UserSecretsClient()
HF_TOKEN = s.get_secret("HF_TOKEN")
HF_REPO = s.get_secret("HF_REPO")
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HF_REPO"] = HF_REPO

try:
    GH_REPO = s.get_secret("GH_REPO")
except Exception:
    GH_REPO = "CHANGE_ME/happy"

try:
    GH_TOKEN = s.get_secret("GH_TOKEN")
except Exception:
    GH_TOKEN = None

url = f"https://{GH_TOKEN}@github.com/{GH_REPO}.git" if GH_TOKEN else f"https://github.com/{GH_REPO}.git"
work = "/kaggle/working/happy"
if not os.path.exists(work):
    subprocess.run(["git", "clone", "--depth", "1", url, work], check=True)
os.chdir(work)
print("repo:", GH_REPO)
print("hf:  ", HF_REPO)
print("torch:", subprocess.run([sys.executable, "-c", "import torch;print(torch.__version__, torch.cuda.is_available())"], capture_output=True, text=True).stdout)


In [ ]:
# 100k отфильтрованных диалоговых пар для bi-encoder.
# В smoke-режиме пропускаем, чтобы не качать 1.4 ГБ ради проверки.
if not SMOKE:
    subprocess.run([sys.executable, "scripts/fetch_data.py", "--max-pairs", "100000"], check=False)
else:
    print("SMOKE: fetch_data пропущен")


In [ ]:
RUN = "smoke" if SMOKE else "kaggle-main"
OUT = "artifacts_smoke" if SMOKE else "artifacts"

cmd = [
    sys.executable, "scripts/train.py",
    "--out", OUT,
    "--run-name", RUN,
    "--hf-repo", HF_REPO,
    "--epochs", "1" if SMOKE else "8",
    "--push-every-epoch",
    "--resume",
]
if SMOKE:
    cmd.append("--smoke")
print(" ".join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
from pathlib import Path
print(Path(OUT, "training_metrics.json").read_text())
print(sorted(p.name for p in Path(OUT).iterdir()))
print("checkpoint sizes, MB:")
for p in sorted(Path(OUT, "checkpoints").glob("*.pt")):
    print(" ", p.name, round(p.stat().st_size / 1e6, 2))


## Если сессия упала

1. Открой **этот же** ноутбук и снова нажми **Save & Run All (Commit)**.
2. `--resume` подтянет `{run}/checkpoints/*.pt` с Hugging Face и продолжит с последней эпохи.
3. Ничего вводить заново не нужно: `run-name` фиксирован (`kaggle-main`).

Проверить, что артефакты на месте, можно локально:

```bash
python scripts/pull_model.py --repo vaprooll/Happy
```

## Важно

- Не меняй `run-name` между запусками, иначе resume начнётся с нуля.
- Не удаляй `artifacts/checkpoints/` внутри `/kaggle/working`.
- Если HF push отключён, в логах будет `hf push: OFF` — тогда артефакты останутся только в output ноутбука.
